1 - Import and Paths

In [1]:
from pathlib import Path

RUNS_DIR = Path("/home/psivakumar/thesis/notebooks/runs")
assert RUNS_DIR.exists(), f"❌ Not found: {RUNS_DIR}"

print("Runs folder:", RUNS_DIR)
run_folders = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
print("Found run folders:", [p.name for p in run_folders])

print("\n--- Searching for likely prediction artifacts ---")
candidates = []
for rf in run_folders:
    for pat in ["*pred*", "*predict*", "*gen*", "*output*", "*result*", "*.json", "*.csv", "*.pkl", "*.pt"]:
        for f in rf.rglob(pat):
            if f.is_file():
                # keep only smallish metadata files (avoid huge checkpoints)
                if f.suffix.lower() in [".json", ".csv", ".pkl", ".pt", ".txt"]:
                    candidates.append(f)

# show top 60 paths
for p in candidates[:60]:
    print(p)

print("\nTotal candidate files found:", len(candidates))

Runs folder: /home/psivakumar/thesis/notebooks/runs
Found run folders: ['iu_xray_vit_gpt2_notebook', 'vit_gpt2_baseline_iu_xray']

--- Searching for likely prediction artifacts ---
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/generation_config.json
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/config.json
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/generation_config.json
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/tokenizer_config.json
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/special_tokens_map.json
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/vocab.json
/home/psivakumar/thesis/notebooks/runs/iu_xray_vit_gpt2_notebook/best/tokenizer.json
/home/psivakumar/thesis/notebooks/runs/vit_gpt2_baseline_iu_xray/best_model/generation_config.json
/home/psivakumar/thesis/notebooks/runs/vit_gpt2_baseline_iu_xray/best_model/config.json
/home/psivak

2 — Load the same 44 test samples (UIDs + image paths + GT)

In [2]:
import json
from pathlib import Path

ANN_R2GEN_FMT = Path("/home/psivakumar/thesis/data/iu_xray/annotation_r2gencmn_2view_r2gencmnformat.json")
IMG_ROOT = Path("/home/psivakumar/thesis/data/iu_xray/images/images_normalized")  # actual images folder

ann = json.load(open(ANN_R2GEN_FMT))
test_samples = ann["test"]   # should be 44
print("Test samples:", len(test_samples))

# Build list with absolute image paths (2 images) + GT report
vit_test = []
missing = 0
for ex in test_samples:
    uid = str(ex["id"])
    rels = ex["image_path"]  # ["images/..png", "images/..png"]
    # convert to absolute paths in images_normalized
    abs_paths = []
    for r in rels:
        fname = Path(r).name
        p = IMG_ROOT / fname
        abs_paths.append(str(p))
        if not p.exists():
            missing += 1
    vit_test.append({"uid": uid, "image_paths": abs_paths, "ground_truth": ex["report"]})

print("Missing image files:", missing)
print("Example:", vit_test[0])

Test samples: 44
Missing image files: 0
Example: {'uid': '1623', 'image_paths': ['/home/psivakumar/thesis/data/iu_xray/images/images_normalized/1623_IM-0405-4004.dcm.png', '/home/psivakumar/thesis/data/iu_xray/images/images_normalized/1623_IM-0405-1001.dcm.png'], 'ground_truth': 'Normal cardiomediastinal contours. No pneumothorax, pleural effusions or focal lung consolidation. IMPRESSION: No acute cardiopulmonary abnormality.'}


3 — Load ViT+GPT2 model from your saved run

In [4]:
import torch
from transformers import VisionEncoderDecoderModel, AutoTokenizer, AutoImageProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

MODEL_DIR = "/home/psivakumar/thesis/notebooks/runs/vit_gpt2_baseline_iu_xray/best_model"

model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
processor = AutoImageProcessor.from_pretrained(MODEL_DIR)

model.eval()
print("Loaded model/tokenizer/processor from:", MODEL_DIR)

Device: cuda
Loaded model/tokenizer/processor from: /home/psivakumar/thesis/notebooks/runs/vit_gpt2_baseline_iu_xray/best_model


4 — Generate predictions (uses both frontal+lateral)

In [5]:
from PIL import Image
import numpy as np

def load_and_concat_two_views(p1, p2):
    im1 = Image.open(p1).convert("RGB")
    im2 = Image.open(p2).convert("RGB")

    # resize both to same height
    h = 224
    im1 = im1.resize((224, h))
    im2 = im2.resize((224, h))

    # concat horizontally
    new_im = Image.new("RGB", (im1.width + im2.width, h))
    new_im.paste(im1, (0, 0))
    new_im.paste(im2, (im1.width, 0))
    return new_im

vit_results = []

gen_kwargs = dict(
    max_new_tokens=128,
    num_beams=3,
    do_sample=False
)

with torch.no_grad():
    for i, ex in enumerate(vit_test):
        uid = ex["uid"]
        p1, p2 = ex["image_paths"]

        img = load_and_concat_two_views(p1, p2)
        pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(DEVICE)

        output_ids = model.generate(pixel_values, **gen_kwargs)
        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

        vit_results.append({
            "uid": uid,
            "ground_truth": ex["ground_truth"],
            "prediction": pred
        })

print("Generated ViT+GPT2 preds:", len(vit_results))
print("Sample:", vit_results[0]["uid"], "\nPR:", vit_results[0]["prediction"][:200])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask

Generated ViT+GPT2 preds: 44
Sample: 1623 
PR: heart andastum normal size cont. lungs clear Noothax ple eff orothax There nouralusion pneumor.
IMION No cardul abnorm. acuteiopmonaryality No findings pulmonaryema Nouralusion eff orusion No osousali


5 — Save as JSON (same schema as R2Gen)

In [6]:
import json
from pathlib import Path

OUT_PATH = Path("/home/psivakumar/thesis/notebooks/runs/vit_gpt2_baseline_iu_xray/preds_test.json")
with open(OUT_PATH, "w") as f:
    json.dump(vit_results, f, indent=2)

print("Saved:", OUT_PATH)

Saved: /home/psivakumar/thesis/notebooks/runs/vit_gpt2_baseline_iu_xray/preds_test.json
